In [1]:
from flask import Flask, request, jsonify
from transformers import pipeline

app = Flask(__name__)

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")


In [ ]:

@app.route('/summarize', methods=['POST'])
def summarize_text():
    try:
        # Get JSON data from the request
        data = request.get_json()

        # Extract parameters
        user_id = data.get("user_id")
        timestamp = data.get("timestamp")
        key_types = data.get("key_types", [])  # List of 3 key types
        raw_transcript = data.get("raw_transcript")

        if not raw_transcript:
            return jsonify({"error": "Transcript is missing"}), 400

        # Summarize text
        summary = summarizer(
            raw_transcript, 
            max_length=350, 
            min_length=150, 
            length_penalty=2.0, 
            num_beams=4, 
            early_stopping=True
        )[0]["summary_text"]

        # Organizing response
        response = {
            "user_id": user_id,
            "timestamp": timestamp,
            "key_types": key_types,
            "summarized_transcript": summary
        }

        return jsonify(response)

    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True)
